# Neural ODEs as a Bridge to Implicit Layers

A neural ODE represents hidden evolution by

$$
\frac{dh(t)}{dt}=v_\theta(h(t),t).
$$

With explicit Euler and a time-independent vector field,

$$
h_{k+1}=h_k+\Delta t\,v_\theta(h_k).
$$

This is not a DEQ solve by itself, but it gives a useful bridge: both ODEs and
DEQs describe computation through an operator applied repeatedly. ODEs track a
finite-time path; DEQs solve for a time-independent equilibrium.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

from silva_networks import resolve_device, SolverConfig

torch.manual_seed(7)
np.random.seed(7)
device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
device

## A Rotating-Damped Vector Field

For a hand-checkable example, let

$$
v(h)=Ah,
\qquad
A=\begin{bmatrix}-0.15 & -1\\ 1 & -0.15\end{bmatrix}.
$$

The first Euler update is

$$
h_1
=h_0+\Delta t Ah_0.
$$

With $h_0=(1,0)$ and $\Delta t=0.1$,

$$
Ah_0=(-0.15,1),
\qquad
h_1=(0.985,0.1).
$$

In [ ]:
from silva_networks import SILVAEulerFlowBlock

class LinearVectorField(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.A = torch.nn.Parameter(torch.tensor([[-0.15, -1.0], [1.0, -0.15]]), requires_grad=False)

    def forward(self, h):
        return h @ self.A.T

ode = SILVAEulerFlowBlock(2, steps=50, step_size=0.1, vector_field=LinearVectorField()).to(device)
h0 = torch.tensor([[1.0, 0.0]], device=device)
terminal, trajectory = ode(h0, return_trajectory=True)
trajectory_xy = trajectory[:, 0].detach().cpu().numpy()
trajectory_xy[:3], terminal.detach().cpu().numpy()

In [ ]:
plt.figure(figsize=(4.2, 4.0))
plt.plot(trajectory_xy[:, 0], trajectory_xy[:, 1], marker="o", markersize=2)
plt.scatter([trajectory_xy[0, 0]], [trajectory_xy[0, 1]], label="start")
plt.scatter([trajectory_xy[-1, 0]], [trajectory_xy[-1, 1]], label="end")
plt.axis("equal")
plt.xlabel("h1")
plt.ylabel("h2")
plt.legend()
plt.tight_layout()

## Trainable ODE Block

`SILVAEulerFlowBlock` accepts a trainable vector field. It is an ordinary
PyTorch module, so gradients flow through every Euler step.

In [ ]:
x = torch.randn(16, 3, device=device)
y = (x[:, 0] > 0).long()
feature = torch.nn.Linear(3, 8).to(device)
flow = SILVAEulerFlowBlock(8, hidden_dim=16, steps=5, step_size=0.15).to(device)
head = torch.nn.Linear(8, 2).to(device)
optim = torch.optim.Adam(list(feature.parameters()) + list(flow.parameters()) + list(head.parameters()), lr=0.02)

losses = []
for _ in range(6):
    optim.zero_grad()
    h = torch.tanh(feature(x))
    logits = head(flow(h))
    loss = torch.nn.functional.cross_entropy(logits, y)
    loss.backward()
    optim.step()
    losses.append(float(loss.detach().cpu()))

losses

In [ ]:
plt.figure(figsize=(4.8, 3.0))
plt.plot(losses, marker="o")
plt.xlabel("training step")
plt.ylabel("loss")
plt.tight_layout()

## Citation and Sources

If this package or notebook is used, cite the software repository:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.0.0. MIT License.
https://github.com/jseluis/silva-networks
```

When the work is connected to the SILVA methodology, cite the SILVA Networks
paper as well:

```text
Jose Luis Lima de Jesus Silva. SILVA Networks as Structured Implicit Layers and
Vector Attractors via Dynamic Interaction Fields. 2026. arXiv:2607.28989.
https://arxiv.org/abs/2607.28989
```

Background sources:

- Deep Implicit Layers tutorial: https://implicit-layers-tutorial.org/
- LocusLab DEQ repository: https://github.com/locuslab/deq
- Deep Equilibrium Models: https://arxiv.org/abs/1909.01377
- Multiscale Deep Equilibrium Models: https://arxiv.org/abs/2006.08656
- Stabilizing Equilibrium Models by Jacobian Regularization: https://arxiv.org/abs/2106.14342

The notebook is adapted to the `silva_networks` public API. It links to the
sources above and keep the examples package-native.